# Zero-shot ReID: float vs quantized ViT

Loads a pretrained Vision Transformer (Google `augreg` ViT-Ti/16, converted to ONNX without PyTorch), extracts embeddings with **no ReID fine-tuning**, and benchmarks person re-identification on **Market-1501** for the float model (ONNX Runtime) and for the quantized model (int16 / int8, integer-only simulation).

> The NumPy engine is a simulator, so this notebook uses a small subset of identities. Raise `MAX_IDS` / `DISTRACTORS` (or set `MAX_IDS = None` for the full test set) for a real benchmark; expect hours for the full gallery.

In [ ]:
import os
import sys

import numpy as np
import onnx
import onnxruntime as ort
from tqdm.auto import tqdm

sys.path.append(os.getcwd())

from quantization import OnnxGraph, QuantContainer, native, reid
import vit_from_npz

print("C++ kernels:", "enabled" if native.available() else
      f"disabled ({native.load_error()})")

## Configuration

In [ ]:
MARKET_ROOT = "Market-1501-v15.09.15"   # folder holding query/ and bounding_box_test/
MARKET_GDRIVE_ID = "0B8-rUzbwVRk0c054eEozWG9COHM"

VARIANT = "ti16"                          # ti16 | s16 | b16
ONNX_PATH = f"vit_{VARIANT}_emb.onnx"
BATCH = 1                                 # static batch size of the exported model

MAX_IDS = 5                               # query identities to keep (None = all)
DISTRACTORS = 50                          # extra gallery images of other identities
CALIB_IMAGES = 16
QUANT_TYPES = ["int16", "int8"]           # int8 is expected to collapse on ViT, see below
SEED = 0

# ViT augreg checkpoints use mean = std = 0.5
MEAN, STD = reid.NORMALIZATION["half"]

## 1. Dataset (Market-1501)

Downloads the public archive from Google Drive with `gdown` if it is not present (153 MB).

In [ ]:
if not os.path.isdir(MARKET_ROOT):
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
    import gdown
    gdown.download(id=MARKET_GDRIVE_ID, output="market1501.zip", quiet=False)
    import zipfile
    with zipfile.ZipFile("market1501.zip") as z:
        z.extractall(".")
print(sorted(os.listdir(MARKET_ROOT)))

In [ ]:
query, gallery = reid.load_split(MARKET_ROOT, max_ids=MAX_IDS,
                                 distractors=DISTRACTORS, seed=SEED)
print(f"query:   {len(query[0])} images, {len(np.unique(query[1]))} identities")
print(f"gallery: {len(gallery[0])} images "
      f"({int((gallery[1] == -1).sum())} junk, {int((gallery[1] == 0).sum())} distractor)")

## 2. Model

`vit_from_npz` downloads Google's public checkpoint and writes a static-batch ONNX graph that outputs the **cls embedding** (no classifier head).

In [ ]:
if not os.path.exists(ONNX_PATH):
    npz = vit_from_npz.download(VARIANT, ".")
    model = vit_from_npz.build_onnx(np.load(npz), batch=BATCH, embedding=True)
    onnx.save(model, ONNX_PATH)
print(f"{ONNX_PATH}: {os.path.getsize(ONNX_PATH) / 1e6:.1f} MB")

graph = OnnxGraph.from_model(ONNX_PATH)
print("input:", [(v.name, v.shape) for v in graph.inputs])
print("graph optimizations applied:", graph.fusions)

SIZE = tuple(graph.inputs[0].shape[2:])
session = ort.InferenceSession(ONNX_PATH)
in_name = session.get_inputs()[0].name

## 3. Features and metrics

In [ ]:
def extract(infer, paths, label):
    return reid.extract_features(
        infer, paths, BATCH, SIZE, MEAN, STD,
        lambda it: tqdm(it, desc=label, leave=False))


def benchmark(infer, label):
    fq = extract(infer, query[0], f"{label} query")
    fg = extract(infer, gallery[0], f"{label} gallery")
    dist = reid.cosine_distance(fq, fg)
    res = reid.evaluate_ranking(dist, query[1], gallery[1], query[2], gallery[2])
    return res, fq, fg


results, feats = {}, {}
results["float"], *feats["float"] = benchmark(
    lambda x: session.run(None, {in_name: x})[0], "float")
results["float"]

## 4. Quantize and benchmark

Calibration uses a few random gallery images. `integer` runs the integer-only path (C++ conv / GEMM kernels, fixed-point requantization).

In [ ]:
rng = np.random.default_rng(SEED)
calib_paths = rng.choice(gallery[0], min(CALIB_IMAGES, len(gallery[0])), replace=False)
calib = [np.stack([reid.load_image(p, SIZE, MEAN, STD)
                   for p in calib_paths[i:i + BATCH]])
         for i in range(0, len(calib_paths) - BATCH + 1, BATCH)]

containers = {}
for qt in QUANT_TYPES:
    c = QuantContainer(ONNX_PATH, quant_type=qt)
    c.calibrate(calib)
    containers[qt] = c
    name = f"{qt} / integer"
    results[name], *feats[name] = benchmark(
        lambda x, c=c: c.run_graph(x, "integer"), name)

In [ ]:
ref_gallery = reid.l2_normalize(feats["float"][1])

print(f"{'model':<22}{'mAP':>8}{'R1':>8}{'R5':>8}{'R10':>8}   cos(float)")
for name, r in results.items():
    cos = "" if name == "float" else (
        f"{np.mean(np.sum(reid.l2_normalize(feats[name][1]) * ref_gallery, 1)):.4f}")
    print(f"{name:<22}{r['mAP'] * 100:8.2f}{r['rank1'] * 100:8.2f}"
          f"{r['rank5'] * 100:8.2f}{r['rank10'] * 100:8.2f}   {cos}")
print(f"\n({results['float']['num_valid_queries']} valid queries)")

### Reading the numbers

* **float** is the zero-shot ceiling of this backbone (it was never trained for ReID; Market images are 128x64 squashed to 224x224).
* **int16** should track float closely (feature cosine ~ 0.9996).
* **int8** with per-tensor min/max calibration collapses on ViTs: the residual stream has massive activations (abs-max ~570 vs 99.9th percentile ~15), so one int8 step is ~4.5. Keeping activations in 16 bit (W8A16) restores accuracy; see the README for the measured breakdown.

## 5. Look at a retrieval

Top-5 gallery matches of the first query for the float model (green border = same identity).

In [ ]:
from PIL import Image, ImageDraw
from IPython.display import display

q = 0
dist = reid.cosine_distance(feats["float"][0][q:q + 1], feats["float"][1])[0]
valid = ~(((gallery[1] == query[1][q]) & (gallery[2] == query[2][q])) | (gallery[1] == -1))
order = [i for i in np.argsort(dist) if valid[i]][:5]

def tile(path, color=None):
    im = Image.open(path).convert("RGB").resize((96, 192))
    if color:
        ImageDraw.Draw(im).rectangle([0, 0, 95, 191], outline=color, width=5)
    return im

tiles = [tile(query[0][q], "blue")] + [
    tile(gallery[0][i], "green" if gallery[1][i] == query[1][q] else "red")
    for i in order]
canvas = Image.new("RGB", (100 * len(tiles), 192), "white")
for k, t in enumerate(tiles):
    canvas.paste(t, (100 * k, 0))
print("query (blue) | top-5 matches; distances:", [round(float(dist[i]), 3) for i in order])
display(canvas)